In [2]:
import os
os.environ["GROQ_API_KEY"]="gsk_"

In [4]:
%pip install -U \
langchain==1.3.14 \
langchain-community \
langchain-huggingface \
langchain-groq \
duckduckgo-search \
ddgs \
requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Succe

In [16]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

In [17]:
import os

os.environ["WEATHERSTACK_API_KEY"] = "7e169860e4b6e7632384cfeb39a49138"

# e333a0fc1f48881bde757dd70fc5c4b6

In [18]:
@tool
def get_weather_data(city: str) -> dict:
    """Fetch current weather for a city."""

    url = f"https://api.weatherstack.com/current?access_key=e333a0fc1f48881bde757dd70fc5c4b6&query={city}"

    response = requests.get(url)

    return response.json()

In [19]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


In [20]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults(
    max_results=5
)

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt="""
    You are a helpful AI assistant.

    You can use the available tools whenever needed.
    Think carefully before choosing a tool.
    If multiple tools are required, use them one at a time.
    """
)

In [22]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": " Find the capital of Madhya Pradesh, then find it's current weather condition."
            }
        ]
    }
)

print(response["messages"][-1].content)
print(response)

The capital of Madhya Pradesh is Bhopal. The current weather condition in Bhopal is Light Rain with a temperature of 24 degrees Celsius and a humidity of 100%. The wind is blowing at a speed of 18 km/h from the west-southwest direction. The air quality is good, with a low level of pollutants. The visibility is 3 km, and the UV index is 1.
{'messages': [HumanMessage(content=" Find the capital of Madhya Pradesh, then find it's current weather condition.", additional_kwargs={}, response_metadata={}, id='19cdb47b-d4e0-4162-9853-1c2d208e2421'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fz07njqxz', 'function': {'arguments': '{"query":"capital of Madhya Pradesh"}', 'name': 'duckduckgo_results_json'}, 'type': 'function'}, {'id': '7qvv1s04t', 'function': {'arguments': '{"city":"Bhopal"}', 'name': 'get_weather_data'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 374, 'total_tokens': 412, 'completion_time': 0.09363158, '

In [23]:
for msg in response["messages"]:
    print(type(msg).__name__)
    print(msg.content)
    print("-" * 50)

HumanMessage
 Find the capital of Madhya Pradesh, then find it's current weather condition.
--------------------------------------------------
AIMessage

--------------------------------------------------
ToolMessage
snippet: After the independence of India, Madhya Pradesh was created in 1950 from the former British Central Provinces and Berar and the princely states of Makrai and Chhattisgarh, with Nagpur as the capital of the state., title: Madhya Pradesh - Wikipedia, link: https://en.wikipedia.org/wiki/Madhya_Pradesh, snippet: Bhopal (Hindi: Bhōpāl, pronounced [bʱoːpaːl] ⓘ) is the capital city of the Indian state of Madhya Pradesh and the administrative headquarters of both Bhopal district and Bhopal division. [10], title: Bhopal - Wikipedia, link: https://en.wikipedia.org/wiki/Bhopal, snippet: Madhya Pradesh, state of India. As its name implies—madhya means “central” and pradesh means “region” or “state”—it is situated in the heart of the country. The state has no coastline and no 